# wrangle / economy / rates

**Intent**
- Gives low, base and high scenarios to 2070, in percent per year, as constants, and must:
  - Give `economy_rates_inflation_{low,base,high}_pct`, the inflation rate.
  - Give `economy_rates_riskfree_{low,base,high}_pct`, the nominal bank rate.
  - Give `economy_rates_real_{low,base,high}_pct`, the real return of cash.
  - Keep real equal to risk-free less inflation, from the same `interest` rows.
  - Be the floor and discount rate of `portfolio/return` and `portfolio/withdrawal`, and the inflation under `household/costs`.

**Data**
- `training` is the 5, 10, 20 and 40 year window means of the bank rate, inflation and real return of cash.
- It is synthetic until the `interest` table is read.
- That table holds the Reserve Bank `Bank`, `Inflation` and `Net` means over 1, 5, 10, 20 and 40 years.
- `golden` is one valid request, then a request with the input missing.

**Status**
- Low, base and high are placeholders, the smallest, the mean and the largest of the window means.
- The Train section asserts that real equals risk-free less inflation.
- It carries no evaluation, so the golden replay is the gate.

**Future**
- Overall: the work still to do before the first release:
  - Promote the target from local to served, then add the Register stage, once the limits are fixed against real data.
  - Add `query_postgres` and read the `Bank` and `Net` windows.
  - Model inflation as an AR(1) pulled toward an anchored Reserve Bank target, with a de-anchored regime for the high scenario.
  - Model the real rate as mean reverting and add inflation, so the pair stays consistent.
  - Simulate to 2070, take compound rates, and set low, base and high as percentiles.
  - Compare the cash rate earned with the published one, and add index linked bond yields as an anchor.
  - Serve the numbers to the planning sheet through `wrangle` and an Apps Script function, which is not yet built.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/economy/rates")

GOLDEN = "economy_rates_golden"

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

WINDOWS = [5, 10, 20, 40]
BANK = [2.0, 2.6, 3.6, 5.0]
INFLATION = [2.9, 2.7, 2.8, 3.24]
collect_run(
    ws,
    1000,
    label="Synthetic window means of the bank rate, inflation and real return of cash",
    name="economy_rates_training",
    source="builder",
    build=lambda: {"window_years": WINDOWS, "bank_pct": BANK, "inflation_pct": INFLATION, "net_pct": [round(bank - inflation, 4) for bank, inflation in zip(BANK, INFLATION, strict=True)]},
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "economy_rates_training"))
print("windows", ws.training.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
bank, inflation, net = (ws.training[column].to_numpy() for column in ("bank_pct", "inflation_pct", "net_pct"))
assert abs(bank - inflation - net).max() < 1e-9, "the real return of cash is not the bank rate less inflation"


def scenarios(series):
    return [round(float(series.min()), 2), round(float(series.mean()), 2), round(float(series.max()), 2)]


ws.params = dict(zip(ws.outputs, [*scenarios(inflation), *scenarios(bank), *scenarios(net)], strict=True))
print("constants", ws.params)
assert list(ws.params.values()) == [2.7, 2.91, 3.24, 2.0, 3.3, 5.0, -0.9, 0.39, 1.76], "the fit did not recover the window summary"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

define_model(ws, model=1000, flavour="static", source=ws.params, label="Window summaries of the bank rate, inflation and real cash return")

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [1.0, None], "expect": ["ok", "error"]}
package_model(ws, 1000, golden, target={"kind": "local"})